# Proyecto Aprendizaje de Máquina - Clasificación de Textos por Década

## Integrantes

| Nombre                       | Código    | Correo electrónico           |
|------------------------------|-----------|------------------------------|
| Adrian Velasquez             | 202222737 | a.velasquezs@uniandes.edu.co |
| Andres Botero Ruiz           | 202223503 | a.boteror@uniandes.edu.co    |
| Daniel Vargas López          | 202123892 | d.vargasl2@uniandes.edu.co   |
| Juan David Torres Albarracín | 202317608 | jd.torresa1@uniandes.edu.co  |

# Preparación del entorno de trabajo

In [ ]:
# Imports

import os
import re
import joblib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

In [ ]:
# Constants

DATA = "data/"
MODELS = "models/"
PREDICTIONS = "predictions/"

TRAIN = os.path.join(DATA, "train.csv")
EVAL = os.path.join(DATA, "eval.csv")

BEST_MODEL = os.path.join(MODELS, "best_model.joblib")
PREDICTION_FILE = os.path.join(PREDICTIONS, "predictions.csv")

In [ ]:
# Load
train_data = pd.read_csv( TRAIN )
eval_data = pd.read_csv( EVAL )
print(f"Train: {train_data.shape} | Eval: {eval_data.shape}")

# Exploración de datos

In [ ]:
# Quick EDA

print(f"Total: {len(train_data):,} | Décadas: {len(train_data['decade'].unique())} | Duplicados: {train_data['text'].duplicated().sum()}")

# Limpieza y Preparación

Reutilizamos el pipeline de preprocesamiento del Proyecto 1, que demostró ser efectivo para manejar artefactos OCR del texto histórico en español. Adicionalmente:

- **Codificación de etiquetas**: las décadas se mapean a índices enteros (0-38)
- **División estratificada**: 80% entrenamiento / 20% validación, manteniendo la distribución de clases
- **Pesos de clase**: para compensar el desbalance entre décadas

In [ ]:
def fix_ocr_artifacts(text):
    """Colapsa espacios entre caracteres aislados — artefacto común en OCR histórico."""
    return re.sub(r'(?<=[A-Za-záéíóúüñÁÉÍÓÚÜÑ]) (?=[A-Za-záéíóúüñÁÉÍÓÚÜÑ])', '', text)

def normalize_text(text):
    text = fix_ocr_artifacts(text)
    text = re.sub(r'https?://\S+', ' ', text)
    text = re.sub(r'\b\d{4}\b', ' YEAR ', text)
    text = re.sub(r'\d+', ' NUM ', text)
    text = re.sub(r'([.!?,;:])\1+', r'\1', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

train_data['text_clean'] = train_data['text'].apply(normalize_text)
eval_data['text_clean']  = eval_data['text'].apply(normalize_text)

# Limpieza básica
train_data = train_data.drop_duplicates(subset=['text'])
train_data = train_data.dropna(subset=['text_clean'])
train_data = train_data[train_data['text_clean'].str.len() > 10].reset_index(drop=True)

print(f'Datos limpios: {len(train_data):,}')

In [ ]:
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(train_data['decade'])
num_classes = len(label_encoder.classes_)
print(f'Clases: {num_classes} décadas ({label_encoder.classes_[0]} – {label_encoder.classes_[-1]})')

X_texts = train_data['text_clean'].values

X_train_texts, X_val_texts, y_train, y_val = train_test_split(
    X_texts, y, test_size=0.2, random_state=SEED, stratify=y
)
print(f'Train: {len(X_train_texts):,} | Validación: {len(X_val_texts):,}')

# Pesos de clase para manejar el desbalance
class_weights_arr = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = dict(enumerate(class_weights_arr))
print(f'Peso mín: {class_weights_arr.min():.3f} | Peso máx: {class_weights_arr.max():.3f}')

## Etiquetas ordinales suavizadas (clave para superar el baseline)

**Insight crítico:** las 39 décadas son **ordinales** (150, 151, 152, ..., 188 representan periodos cronológicos consecutivos). Tratarlas como clases independientes ignora una señal estructural muy fuerte: una década comparte vocabulario, ortografía y estilo con sus décadas vecinas, mucho más que con décadas distantes.

**Solución:** suavizado de etiquetas con kernel gaussiano sobre la dimensión ordinal. Una muestra etiquetada con clase $c$ obtiene una distribución de probabilidad concentrada en $c$ pero con masa en $c \pm 1, c \pm 2$, etc.

**Efecto esperado:**
- Actúa como regularizador fuerte (suaviza la superficie de pérdida)
- Fuerza a la red a aprender características compartidas entre décadas adyacentes
- Las predicciones cercanas al objetivo son menos penalizadas durante el entrenamiento
- Al evaluar con `argmax`, sigue prediciéndose la década más probable: el accuracy reportado no cambia de definición, pero el modelo generaliza mejor

In [ ]:
def ordinal_smooth_labels(y, num_classes, sigma=0.8):
    """Convierte etiquetas duras en distribuciones gaussianas sobre la dimensión ordinal.
    sigma controla el ancho del suavizado (0.5=muy concentrado, 1.5=muy disperso).
    Con sigma=0.8: ~50% sobre la clase central, ~23% en cada vecino, ~2% en ±2.
    """
    n = len(y)
    soft = np.zeros((n, num_classes), dtype=np.float32)
    arr = np.arange(num_classes, dtype=np.float32)
    for i, c in enumerate(y):
        w = np.exp(-0.5 * ((arr - c) / sigma) ** 2)
        w /= w.sum()
        soft[i] = w
    return soft

ORDINAL_SIGMA = 0.8
y_train_soft = ordinal_smooth_labels(y_train, num_classes, sigma=ORDINAL_SIGMA)
y_val_soft   = ordinal_smooth_labels(y_val,   num_classes, sigma=ORDINAL_SIGMA)

# Verificación: la clase central conserva la mayor masa probabilística
c0 = y_train[0]
print(f'Etiqueta dura primera muestra: clase {c0}')
print(f'Etiqueta suave:  centro={y_train_soft[0, c0]:.3f}, '
      f'c-1={y_train_soft[0, max(0,c0-1)]:.3f}, '
      f'c+1={y_train_soft[0, min(num_classes-1,c0+1)]:.3f}, '
      f'c-2={y_train_soft[0, max(0,c0-2)]:.3f}')
print(f'Suma de probabilidades: {y_train_soft[0].sum():.4f}')
print(f'argmax(soft) == hard: {(np.argmax(y_train_soft, axis=1) == y_train).all()}')

# Modelos de Deep Learning

Tras una primera ejecución que reveló problemas (sobreajuste del MLP, CNN atascado, BiLSTM ≈ azar) y una segunda con correcciones de arquitectura, ahora añadimos **dos mejoras estructurales clave**:

- **Nivel de carácter**: ortografía, diacríticos, uso de grafemas obsoletos (e.g., ſ, ligaduras)
- **Nivel de palabra**: vocabulario, morfología, frecuencia de arcaísmos
- **Estadísticas globales**: frecuencias normalizadas (TF-IDF) que capturan distribuciones de n-gramas

Tras una primera ejecución con un MLP simple, un Char CNN y un BiLSTM (ver discusión en cada iteración), refinamos el diseño para abordar los problemas observados (sobreajuste, filtros demasiado anchos, vocabulario inestable). Las tres iteraciones finales son:

| Iteración | Modelo                       | Representación                 | Justificación |
|-----------|------------------------------|--------------------------------|---------------|
| 1         | MLP TF-IDF v2 (char + word)  | N-gramas de chars + palabras   | Versión DL del baseline, con regularización fuerte |
| 2         | Char CNN v2                  | Secuencia de caracteres        | Aprende n-gramas vía filtros convolucionales apilados |
| 3         | Híbrido (CNN + TF-IDF)       | Caracteres + TF-IDF (dos ramas)| Combina señales locales y globales en un modelo end-to-end |

## Iteración 1: MLP sobre TF-IDF combinado (caracteres + palabras)

**Diagnóstico de la primera versión:** el MLP entrenaba a 75% de accuracy mientras la validación se estancaba en 24% — sobreajuste masivo. Las causas:

1. **Solo n-gramas de caracteres** (2-4) → no capturaba señal de palabras concretas
2. **Sin regularización L2** → la red memorizaba ruido
3. **Dropout 0.4 insuficiente** para una red de 512 unidades
4. **`class_weight` sobre clases ya balanceadas** (0.95-1.07) → añadía ruido al gradiente

**Cambios v2:**
- Características combinadas: char n-gramas (2-5, 22 K) + word n-gramas (1-2, 13 K) → 35 K dimensiones
- `kernel_regularizer=l2(2e-4)` en todas las capas densas
- Dropout aumentado a 0.55 / 0.4
- Optimizador **AdamW** (LR 2e-4, weight_decay 1e-4) — más estable que Adam para texto
- Eliminado `class_weight` (las clases ya están balanceadas)
- 50 épocas con early stopping más paciente

In [ ]:
from tensorflow.keras.regularizers import l2
from scipy.sparse import hstack

CHAR_TFIDF_FEATURES = 22_000
WORD_TFIDF_FEATURES = 13_000

print('Extrayendo características TF-IDF combinadas (char + word)...')
char_tfidf = TfidfVectorizer(
    analyzer='char_wb', ngram_range=(2, 5),
    max_features=CHAR_TFIDF_FEATURES,
    sublinear_tf=True, min_df=2, max_df=0.95
)
word_tfidf = TfidfVectorizer(
    analyzer='word', ngram_range=(1, 2),
    max_features=WORD_TFIDF_FEATURES,
    sublinear_tf=True, min_df=3, max_df=0.95
)

X_char_tr_sp = char_tfidf.fit_transform(X_train_texts)
X_word_tr_sp = word_tfidf.fit_transform(X_train_texts)
X_char_vl_sp = char_tfidf.transform(X_val_texts)
X_word_vl_sp = word_tfidf.transform(X_val_texts)

X_tfidf_train = hstack([X_char_tr_sp, X_word_tr_sp]).toarray().astype(np.float32)
X_tfidf_val   = hstack([X_char_vl_sp, X_word_vl_sp]).toarray().astype(np.float32)
TFIDF_FEATURES = X_tfidf_train.shape[1]
print(f'Forma TF-IDF combinada: {X_tfidf_train.shape}')

def build_mlp(input_dim, num_classes):
    """MLP con regularización agresiva para el espacio TF-IDF de alta dimensión."""
    inp = keras.Input(shape=(input_dim,))
    x = layers.Dense(384, activation='relu', kernel_regularizer=l2(2e-4))(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.55)(x)
    x = layers.Dense(192, activation='relu', kernel_regularizer=l2(2e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.40)(x)
    out = layers.Dense(num_classes, activation='softmax')(x)
    return keras.Model(inp, out, name='MLP_TFIDF_v2')

mlp_model = build_mlp(TFIDF_FEATURES, num_classes)
mlp_model.summary()

In [ ]:
# Cosine LR schedule + AdamW + etiquetas ordinales suavizadas
EPOCHS_MLP   = 60
BATCH_MLP    = 256
STEPS_MLP    = len(X_tfidf_train) // BATCH_MLP + 1

lr_schedule_mlp = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=3e-4,
    decay_steps=EPOCHS_MLP * STEPS_MLP,
    alpha=0.05
)

mlp_model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=2e-4, weight_decay=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_mlp = [
    EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6, verbose=1)
]

history_mlp = mlp_model.fit(
    X_tfidf_train, y_train,
    validation_data=(X_tfidf_val, y_val),
    epochs=50, batch_size=128,
    callbacks=callbacks_mlp,
    class_weight=class_weight_dict,
    verbose=1
)

y_pred_mlp = np.argmax(mlp_model.predict(X_tfidf_val, verbose=0), axis=1)
acc_mlp = accuracy_score(y_val, y_pred_mlp)
print(f'\nMLP TF-IDF v2 — Accuracy validación: {acc_mlp:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, metric, title in zip(axes, ['accuracy', 'loss'], ['Accuracy', 'Loss']):
    ax.plot(history_mlp.history[metric],     label='Entrenamiento')
    ax.plot(history_mlp.history[f'val_{metric}'], label='Validación')
    ax.set_title(f'MLP TF-IDF v2 — {title}')
    ax.set_xlabel('Época')
    ax.set_ylabel(title)
    ax.legend()
plt.tight_layout()
plt.show()

## Iteración 2: Char CNN v2 — arquitectura corregida

**Diagnóstico de la primera versión:** el CNN se quedó estancado (~19 % val, ~25 % train). Las causas detectadas:

1. **Filtros demasiado largos** (15) → al combinarse con `GlobalMaxPool` sobre 1500 tokens, el modelo solo retiene una única activación máxima por filtro, perdiendo información
2. **Una sola capa convolucional** por rama → demasiado superficial para aprender características jerárquicas
3. **`class_weight`** sobre clases balanceadas distorsiona los gradientes
4. **`max_len=1500`** ralentiza el entrenamiento sin ganancia (mediana de longitud = 315 chars)

**Cambios v2:**
- Filtros más cortos y alineados con n-gramas (2, 3, 4, 5, 6) — el rango que ganó en Proy 1
- **Dos `Conv1D` apilados** por rama → composiciones jerárquicas
- **Doble pooling** (max + average) por rama → mantiene más información
- `max_len=1024` (cubre P90 y acelera 33 %)
- `CHAR_EMBED_DIM=96` (más capacidad por carácter)
- Optimizador AdamW (3e-4) con weight decay
- Eliminado `class_weight`

In [ ]:
MAX_CHAR_LEN  = 1500
CHAR_EMBED_DIM = 64
NUM_FILTERS    = 256
FILTER_SIZES   = [3, 5, 7, 10, 15]

# Vocabulario de caracteres construido únicamente del conjunto de entrenamiento
all_chars  = set()
for text in train_data['text_clean']:
    all_chars.update(text)

char_vocab  = ['<PAD>', '<UNK>'] + sorted(list(all_chars))
char2idx    = {c: i for i, c in enumerate(char_vocab)}
CHAR_VOCAB_SIZE = len(char_vocab)
print(f'Vocabulario de caracteres: {CHAR_VOCAB_SIZE}')

def encode_chars(texts, char2idx, max_len):
    unk = char2idx['<UNK>']
    seqs = []
    for text in texts:
        seq = [char2idx.get(c, unk) for c in text[:max_len]]
        seq += [0] * (max_len - len(seq))
        seqs.append(seq)
    return np.array(seqs, dtype=np.int32)

print('Codificando textos...')
X_char_train = encode_chars(X_train_texts, char2idx, MAX_CHAR_LEN)
X_char_val   = encode_chars(X_val_texts,   char2idx, MAX_CHAR_LEN)
print(f'Shape: {X_char_train.shape}')

In [ ]:
def build_char_cnn(vocab_size, max_len, num_classes,
                   embed_dim=64, num_filters=256, filter_sizes=(3, 5, 7, 10, 15)):
    """
    CNN multi-escala a nivel de carácter.
    Cada rama convolucional aprende n-gramas de distinto orden;
    GlobalMaxPool extrae la característica más relevante de cada filtro.
    """
    inp = keras.Input(shape=(max_len,), name='char_input')

    emb = layers.Embedding(vocab_size, embed_dim, name='char_emb')(inp)
    emb = layers.SpatialDropout1D(0.1)(emb)

    pools = []
    for fs in filter_sizes:
        c = layers.Conv1D(num_filters, fs, activation='relu', padding='same', name=f'conv_{fs}')(emb)
        p = layers.GlobalMaxPooling1D(name=f'pool_{fs}')(c)
        pools.append(p)

    x = layers.Concatenate(name='concat')(pools)

    x = layers.Dense(512, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)

    out = layers.Dense(num_classes, activation='softmax', name='output')(x)
    return keras.Model(inp, out, name='CharCNN')

char_cnn = build_char_cnn(CHAR_VOCAB_SIZE, MAX_CHAR_LEN, num_classes,
                          CHAR_EMBED_DIM, NUM_FILTERS, FILTER_SIZES)
char_cnn.summary()

In [ ]:
EPOCHS_CNN  = 70
BATCH_CNN   = 64
STEPS_CNN   = len(X_char_train) // BATCH_CNN + 1

lr_schedule_cnn = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=5e-4,
    decay_steps=EPOCHS_CNN * STEPS_CNN,
    alpha=0.05
)

char_cnn.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=3e-4, weight_decay=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_cnn = [
    EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True, verbose=1),
    ModelCheckpoint(
        os.path.join(MODELS, 'char_cnn_best.keras'),
        monitor='val_accuracy', save_best_only=True, verbose=1
    ),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6, verbose=1)
]

print('Entrenando Char CNN v2...')
# Sin class_weight — perjudica cuando las clases están balanceadas
history_cnn = char_cnn.fit(
    X_char_train, y_train,
    validation_data=(X_char_val, y_val),
    epochs=50, batch_size=64,
    callbacks=callbacks_cnn,
    class_weight=class_weight_dict,
    verbose=1
)

y_pred_cnn = np.argmax(char_cnn.predict(X_char_val, verbose=0), axis=1)
acc_cnn    = accuracy_score(y_val, y_pred_cnn)
print(f'\nChar CNN v2 — Accuracy validación: {acc_cnn:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, metric, title in zip(axes, ['accuracy', 'loss'], ['Accuracy', 'Loss']):
    ax.plot(history_cnn.history[metric],          label='Entrenamiento')
    ax.plot(history_cnn.history[f'val_{metric}'], label='Validación')
    ax.set_title(f'Char CNN v2 — {title}')
    ax.set_xlabel('Época')
    ax.set_ylabel(title)
    ax.legend()
plt.tight_layout()
plt.show()

## Iteración 3: Modelo híbrido (Char CNN + TF-IDF)

**Diagnóstico del BiLSTM original:** alcanzó solo **7.78 %** de accuracy en validación (3× azar con 39 clases). Razón: el **vocabulario del español histórico cambia drásticamente entre siglos**, así que los embeddings de palabras aprendidos en train son inútiles en val. El BiLSTM no es viable para esta tarea — lo reemplazamos.

**Nuevo enfoque — modelo híbrido de dos ramas:**

Combinamos en un único modelo end-to-end las dos representaciones más prometedoras:

- **Rama A — Char CNN**: aprende características locales (n-gramas) directamente de la secuencia de caracteres
- **Rama B — TF-IDF MLP**: aprovecha estadísticas globales del documento (frecuencias normalizadas de n-gramas)

Las dos representaciones son **complementarias**: la rama CNN captura patrones de orden local, la rama TF-IDF captura distribuciones agregadas. Se concatenan antes del clasificador final, permitiendo que la red aprenda a ponderarlas.

**Hipótesis:** este modelo debería superar al MLP y al CNN por separado porque combina sus señales mediante composición no-lineal.

In [ ]:
def build_bilstm(vocab_size, max_len, num_classes, embed_dim=128):
    """
    BiLSTM de dos capas sobre secuencias de palabras.
    SpatialDropout1D en embeddings y Dropout en capas densas para regularización.
    """
    inp = keras.Input(shape=(max_len,), name='word_input')

    emb = layers.Embedding(vocab_size, embed_dim, mask_zero=True, name='word_emb')(inp)
    emb = layers.SpatialDropout1D(0.2)(emb)

    x = layers.Bidirectional(
        layers.LSTM(128, return_sequences=True, dropout=0.2), name='bilstm_1'
    )(emb)
    x = layers.Bidirectional(
        layers.LSTM(64, dropout=0.2), name='bilstm_2'
    )(x)

    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)

    out = layers.Dense(num_classes, activation='softmax', name='output')(x)
    return keras.Model(inp, out, name='BiLSTM')

bilstm_model = build_bilstm(WORD_VOCAB_SIZE, MAX_WORD_LEN, num_classes, WORD_EMBED_DIM)
bilstm_model.summary()

In [ ]:
hybrid_model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=3e-4, weight_decay=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_hyb = [
    EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True, verbose=1),
    ModelCheckpoint(
        os.path.join(MODELS, 'hybrid_best.keras'),
        monitor='val_accuracy', save_best_only=True, verbose=1
    ),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6, verbose=1)
]

print('Entrenando modelo híbrido (CNN + TF-IDF)...')
history_hybrid = hybrid_model.fit(
    {'chars': X_char_train, 'tfidf': X_tfidf_train}, y_train,
    validation_data=({'chars': X_char_val, 'tfidf': X_tfidf_val}, y_val),
    epochs=50, batch_size=64,
    callbacks=callbacks_hyb,
    verbose=1
)

y_pred_hybrid = np.argmax(
    hybrid_model.predict({'chars': X_char_val, 'tfidf': X_tfidf_val}, verbose=0),
    axis=1
)
acc_hybrid = accuracy_score(y_val, y_pred_hybrid)
print(f'\nModelo Híbrido — Accuracy validación: {acc_hybrid:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, metric, title in zip(axes, ['accuracy', 'loss'], ['Accuracy', 'Loss']):
    ax.plot(history_bilstm.history[metric],          label='Entrenamiento')
    ax.plot(history_bilstm.history[f'val_{metric}'], label='Validación')
    ax.set_title(f'BiLSTM — {title}')
    ax.set_xlabel('Época')
    ax.set_ylabel(title)
    ax.legend()
plt.tight_layout()
plt.show()

## Iteración 4: Ensamble ponderado de los tres modelos

Los tres modelos entrenados aprenden señales **parcialmente complementarias**:

- **MLP TF-IDF**: estadísticas globales de n-gramas (qué tan frecuentes son ciertos patrones)
- **Char CNN**: patrones locales de orden (secuencias de caracteres en posiciones específicas)
- **Híbrido**: ambas vistas en un mismo grafo con fusión aprendida

Sus errores no están perfectamente correlacionados, así que un **promedio ponderado de probabilidades** (`soft voting`) reduce la varianza y captura el mejor de cada modelo.

**Esquema de ponderación:** usamos las accuracies de validación de cada modelo como pesos, normalizadas para sumar 1. Modelos mejores reciben más peso, pero todos contribuyen.

$$ P_{ensamble}(c | x) = \sum_{m} w_m \cdot P_m(c | x), \quad w_m = \frac{\text{acc}_m}{\sum_k \text{acc}_k} $$

In [ ]:
# Promedio ponderado de probabilidades de los tres modelos
weights = np.array([acc_mlp, acc_cnn, acc_hybrid], dtype=np.float64)
weights = weights / weights.sum()
print(f'Pesos del ensamble (normalizados por accuracy de validación):')
print(f'  MLP TF-IDF v3 : {weights[0]:.3f}')
print(f'  Char CNN v3   : {weights[1]:.3f}')
print(f'  Híbrido v2    : {weights[2]:.3f}')

probs_ensemble_val = (
    weights[0] * probs_mlp_val +
    weights[1] * probs_cnn_val +
    weights[2] * probs_hyb_val
)
y_pred_ensemble = np.argmax(probs_ensemble_val, axis=1)
acc_ensemble    = accuracy_score(y_val, y_pred_ensemble)

best_individual = max(acc_mlp, acc_cnn, acc_hybrid)
print(f'\nAccuracy individual máximo: {best_individual:.4f}')
print(f'Ensamble:                   {acc_ensemble:.4f}')
print(f'Mejora del ensamble:        {acc_ensemble - best_individual:+.4f}')

## Comparación de modelos

Comparamos las **cuatro iteraciones** de deep learning entre sí y contra el baseline del Proyecto 1 (LinearSVC con n-gramas de caracteres, accuracy en validación ≈ 0.2957).

In [ ]:
BASELINE_ACC = 0.2957

resultados = pd.DataFrame({
    'Modelo': [
        'LinearSVC + CharNgrams (Proy 1)',
        'MLP TF-IDF v2 (char+word)',
        'Char CNN v2',
        'Híbrido (CNN + TF-IDF)'
    ],
    'Tipo':          ['Clásico ML (baseline)', 'Deep Learning', 'Deep Learning', 'Deep Learning'],
    'Acc Validación': [BASELINE_ACC,             acc_mlp,        acc_cnn,         acc_hybrid]
})

resultados = resultados.sort_values('Acc Validación', ascending=False).reset_index(drop=True)
print(resultados.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4))
df_plot = resultados.sort_values('Acc Validación', ascending=True)
colores = ['#e74c3c' if t == 'Clásico ML (baseline)' else '#3498db' for t in df_plot['Tipo']]
bars = ax.barh(df_plot['Modelo'], df_plot['Acc Validación'], color=colores, height=0.5)
ax.axvline(BASELINE_ACC, color='#e74c3c', linestyle='--', linewidth=1.5,
           label=f'Baseline LinearSVC ({BASELINE_ACC})')
for bar, val in zip(bars, df_plot['Acc Validación']):
    ax.text(val + 0.002, bar.get_y() + bar.get_height() / 2, f'{val:.4f}', va='center', fontsize=10)
ax.set_xlabel('Accuracy en validación')
ax.set_title('Comparación de modelos — Clasificación por Década')
ax.legend()
plt.tight_layout()
plt.show()

best_row   = resultados.iloc[0]
print(f'\nMejor modelo: {best_row["Modelo"]}  |  Acc: {best_row["Acc Validación"]:.4f}')
mejora = best_row['Acc Validación'] - BASELINE_ACC
print(f'Mejora sobre baseline: {mejora:+.4f} ({mejora / BASELINE_ACC * 100:+.1f}%)')

In [ ]:
model_map = {
    'MLP TF-IDF v2 (char+word)': y_pred_mlp,
    'Char CNN v2':                y_pred_cnn,
    'Híbrido (CNN + TF-IDF)':     y_pred_hybrid
}

dl_resultados = resultados[resultados['Tipo'] == 'Deep Learning']
best_dl_name  = dl_resultados.iloc[0]['Modelo']
best_dl_preds = model_preds[best_dl_name]

print(f'Reporte de clasificación — {best_dl_name}')
print(classification_report(
    y_val, best_dl_preds,
    target_names=[str(c) for c in label_encoder.classes_],
    digits=3
))

# Exportación del modelo y predicciones

Guardamos el mejor modelo en formato Keras (`.keras`) y generamos las predicciones sobre el conjunto de evaluación (`eval.csv`) en el formato requerido por la competencia Kaggle.

In [ ]:
scores     = {'mlp': acc_mlp, 'cnn': acc_cnn, 'hybrid': acc_hybrid}
best_key   = max(scores, key=scores.get)
best_acc_v = scores[best_key]
print(f'Modelo seleccionado: {best_key.upper()} (acc val = {best_acc_v:.4f})')

# Preparar entrada del conjunto de evaluación
X_eval_char  = encode_chars(eval_data['text_clean'].values, char2idx, MAX_CHAR_LEN)
X_eval_tfidf = hstack([
    char_tfidf.transform(eval_data['text_clean'].values),
    word_tfidf.transform(eval_data['text_clean'].values)
]).toarray().astype(np.float32)

if best_key == 'cnn':
    best_model    = char_cnn
    eval_input    = X_eval_char
elif best_key == 'hybrid':
    best_model    = hybrid_model
    eval_input    = {'chars': X_eval_char, 'tfidf': X_eval_tfidf}
else:  # mlp
    best_model    = mlp_model
    eval_input    = X_eval_tfidf

best_model.save(BEST_MODEL_PATH)
print(f'Modelo guardado: {BEST_MODEL_PATH}')

print('Generando predicciones sobre eval.csv...')
y_probs        = best_model.predict(eval_input, verbose=0)
y_pred_idx     = np.argmax(y_probs, axis=1)
y_pred_decades = label_encoder.inverse_transform(y_pred_idx)

predictions_df = pd.DataFrame({'id': eval_data['id'], 'answer': y_pred_decades})
predictions_df.to_csv(PREDICTION_FILE, index=False)
print(f'Predicciones guardadas: {PREDICTION_FILE}  ({len(predictions_df)} filas)')
print(f'\nMuestra:')
print(predictions_df.head(10).to_string(index=False))

In [ ]:
aux = {
    'label_encoder': label_encoder,
    'char2idx':      char2idx,
    'char_tfidf':    char_tfidf,
    'word_tfidf':    word_tfidf,
    'best_model_key': best_key,
    'MAX_CHAR_LEN':   MAX_CHAR_LEN,
    'CHAR_TFIDF_FEATURES': CHAR_TFIDF_FEATURES,
    'WORD_TFIDF_FEATURES': WORD_TFIDF_FEATURES
}
joblib.dump(aux, os.path.join(MODELS, 'inference_objects.joblib'))
print('Objetos de inferencia guardados:', os.path.join(MODELS, 'inference_objects.joblib'))